##Imports

In [0]:
from pyspark.sql.functions import col, trim
from pyspark.sql import SparkSession
from pyspark.dbutils import DBUtils
from datetime import datetime
from io import BytesIO
import pandas as pd
import boto3
import os

## Read Parameters

In [0]:

spark = SparkSession.builder.getOrCreate()
dbutils = DBUtils(spark)

# Function to get AWS credentials from Databricks secrets
def get_aws_credentials(secrets_scope, key_name):
    try:
        # Databricks secrets configuration
   
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        # if aws_secret_key is not None:
        #     print("AWS secret key retrieved successfully.")
        # else:
        #     print("AWS secret key is None. Please check Databricks secrets configuration.")
 
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None

# Function to get AWS credentials from Databricks secrets
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration
 
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
   
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

# Function to connect to AWS and access SSM parameter
def connect_to_aws_and_access_ssm(ssm_key, ssm_client, aws_secret_key):
 
    if aws_secret_key is None:
        print("AWS secret key not available.")
        return
 
    # Access SSM parameter /cfn/a360/reports_output_path
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )
 
    return response['Parameter']['Value']

# Function to get the environment prefix
def get_env_prefix():
    secrets_scope = "aws-databricks"
    aws_keys = get_scope_key_names(secrets_scope)
    aws_key = aws_keys[0] if aws_keys else None
    # Get AWS secret key from Databricks secrets
    aws_secret_key = get_aws_credentials(secrets_scope,aws_key)
    # Connect to AWS using boto3
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                            aws_access_key_id=aws_key,
                            aws_secret_access_key=aws_secret_key)
 
    S3_client = boto3.client('s3', region_name='eu-central-1',
                            aws_access_key_id=aws_key,
                            aws_secret_access_key=aws_secret_key)  

    env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix', ssm_client, aws_secret_key)
    return env_prefix


In [0]:
env = get_env_prefix()

In [0]:
Businessdate = spark.sql(f"""                    
                                select PreviousBusinessDay
                                from `{env}-cdh`.reference.ref_calendar
                                where CalendarDate = cast(getdate() as date)
                            """).first()[0]

PreviousBusinessDay = spark.sql(f"""                    
                                select PreviousBusinessDay
                                from `{env}-cdh`.reference.ref_calendar
                                where CalendarDate = '{Businessdate}'
                            """).first()[0]

In [0]:
def export_excel_to_volume(subfolder):
    # Structure the Export Path
    formatted_date = datetime.now().strftime("%Y%m%d")
    output_path = f"/Volumes/{env}-cdh/landed/customerspecialcases/{subfolder}/{formatted_date}"

    # Generate a unique filename to avoid overwriting
    base_filename = f"{special_cases_table}.xlsx"
    full_export_output = os.path.join(output_path, base_filename)

    counter = 1
    while os.path.exists(full_export_output):
        base_filename = f"{special_cases_table} ({counter}).xlsx"
        full_export_output = os.path.join(output_path, base_filename)
        counter += 1

    # Ensure the directory exists, create it if it doesn't
    os.makedirs(os.path.dirname(full_export_output), exist_ok=True)

    %pip install openpyxl

    # Create an Excel writer object
    output_buffer = BytesIO()
    with pd.ExcelWriter(output_buffer, engine='openpyxl') as writer:
        # Save the subset to a temporary CSV file
        temp_file = os.path.join(output_path, f"CustomersSpecialCases.csv")
        new_init_extfile_scinput.coalesce(1).write.csv(temp_file, header=True, mode="overwrite")
        
        # Read the CSV back into Pandas
        csv_files = [f for f in os.listdir(temp_file) if f.endswith(".csv")]
        if csv_files:
            pandas_df = pd.read_csv(os.path.join(temp_file, csv_files[0]))
            
            # Write to an Excel sheet
            pandas_df.to_excel(writer, index=False)
        for file in os.listdir(temp_file):
            os.remove(os.path.join(temp_file, file))

    with open(full_export_output, "wb") as f:
        f.write(output_buffer.getvalue())

## Read Tables

In [0]:
extfile_customer_special_cases = spark.sql(f"""
                                                select *
                                                from `{env}-cdh`.landed.extfile_customer_special_cases
                                                where businessdate = '{PreviousBusinessDay}'
                                            """)

extfile_customer_special_cases.createOrReplaceTempView("extfile_customer_special_cases")

In [0]:
# List all tables in the schema
tables_df = spark.sql(f"SHOW TABLES IN `{env}-reportinputfiles`.internal")

# Filter for tables that match 'special_cases_%'
stop_run = tables_df.filter(tables_df.tableName.startswith("special_cases_")).count() < 1

if stop_run:
    special_cases = spark.sql(f"""
                                SELECT DISTINCT 
                                          cdi
                                        , tax_number
                                        , Special_case
                                        , update_date
                                        , Grouped_Special_Case
                                        , '{Businessdate}' BusinessDate
                                        , getdate() LoadingDateTime
                                FROM extfile_customer_special_cases
                            """)
    
    special_cases.createOrReplaceTempView("special_cases")

    table_path = f"`{env}-cdh`.landed.extfile_customer_special_cases"
    special_cases.write.mode("append").saveAsTable(table_path)

    dbutils.notebook.exit("No table found matching 'special_cases_%'")

In [0]:
special_cases_table = tables_df.filter(tables_df.tableName.startswith('special_cases_')) \
                               .select(tables_df.tableName) \
                               .orderBy(tables_df.tableName.desc()) \
                               .first()[0]
print(f"Table found: {special_cases_table}")

In [0]:
new_init_extfile_scinput = spark.sql(f"""
                                    select distinct
                                              BorrowerNep
                                            , SpecialCases
                                    from `{env}-reportinputfiles`.internal.`{special_cases_table}`
                                """)

new_init_extfile_scinput.createOrReplaceTempView("new_init_extfile_scinput")

new_extfile_scinput = spark.sql(f"""
                                    select distinct
                                              BorrowerNep
                                            , SpecialCases `Special Cases`
                                            , ROW_NUMBER() OVER (ORDER BY BorrowerNep) AS Id
                                            , '{Businessdate}' BusinessDate
                                            , getdate() LoadingDateTime
                                    from new_init_extfile_scinput
                                """)

new_extfile_scinput.createOrReplaceTempView("new_extfile_scinput")

In [0]:
current_extfile_scinput = spark.sql(f"""
                                        select *
                                        from `{env}-cdh`.landed.extfile_scinput
                                        where businessdate = '{PreviousBusinessDay}'
                                    """)

current_extfile_scinput.createOrReplaceTempView("current_extfile_scinput")

In [0]:
category_mapping = spark.sql(f"""
                                select *
                                from `{env}-cdh`.reference.grouped_special_cases_category_mapping
                            """)

category_mapping.createOrReplaceTempView("category_mapping")

## Validate Date

In [0]:
# Extract from character 15 (index 14) to the end
date_part = special_cases_table[14:]
date_clean = date_part.replace("-", "")

# Validate date format
try:
    date_part = datetime.strptime(date_clean, '%Y%m%d').date()
except ValueError:
    export_excel_to_volume('EXCEPTION')
    raise ValueError(f"Invalid date format extracted from filename: '{date_part}'")

## Column Name Check 

In [0]:
curr_cols = [c.lower() for c in current_extfile_scinput.columns]
new_cols  = [c.lower() for c in new_extfile_scinput.columns]

# Validation
count = (len(new_cols) == 5)
names_subset = set(new_cols) == set(curr_cols)

if not (count and names_subset):
    export_excel_to_volume('EXCEPTION')
    raise ValueError("Data validation failed")

##Special Cases Check 

In [0]:
new_vals = new_extfile_scinput.select(trim(col("Special Cases"))).distinct()
map_vals = category_mapping.select(trim(col("Category"))).distinct()

# find any that aren’t in the mapping
sub = new_vals.subtract(map_vals)

if sub.count() > 0:
    export_excel_to_volume('EXCEPTION')
    raise ValueError("Job failed due to invalid Special Cases.")

In [0]:
special_cases = spark.sql(f"""
                            SELECT DISTINCT 
                                        BorrowerNep AS cdi
                                      , CAST(NULL AS STRING) AS tax_number
                                      , `Special Cases` AS Special_case
                                      , CAST('{date_part}' AS DATE) AS update_date
                                      , b.Grouped AS Grouped_Special_Case
                                      , '{Businessdate}' BusinessDate
                                      , getdate() LoadingDateTime
                            FROM new_extfile_scinput a
                            INNER JOIN category_mapping b
                            ON a.`Special Cases` = b.Category
                        """)

special_cases.createOrReplaceTempView("special_cases")

## Deploy

In [0]:
scinput_table_path = f"`{env}-cdh`.landed.extfile_scinput"
special_cases_table_path = f"`{env}-cdh`.landed.extfile_customer_special_cases"

new_extfile_scinput.write.mode("append").saveAsTable(scinput_table_path)
special_cases.write.mode("append").saveAsTable(special_cases_table_path)

export_excel_to_volume('ARCHIVE')
# spark.sql(f"DROP TABLE `{env}-reportinputfiles`.internal.`{special_cases_table}`")